# 实验一：代码审查意见挖掘与需求理解

## 实验目标
1. 从 GitHub 获取 5 个开源项目的 Pull Request 数据
2. 提取 Code Review、Commit、文件修改等信息
3. 提取特征（AI Reviewer、AI 生成代码检测等）
4. 数据统计分析与可视化

## 准备工作

### ⚠️ 重要：请在运行前配置你的 GitHub Token

1. 打开 `config.py` 文件
2. 将 `GITHUB_TOKEN = "your_github_token_here"` 替换为你的真实 Token
   - 在 GitHub Settings → Developer Settings → Personal Access Tokens → Tokens (classic) 生成
   - 不需要任何权限（public repo 只读即可）

### 安装依赖

In [1]:
# 安装依赖包（首次运行需要）
import sys
!{sys.executable} -m pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


---
## 步骤一：获取 Pull Request 数据

遍历 5 个目标项目，每个项目获取 300 条 PR 数据。

In [2]:
# 导入模块
import sys
import os
sys.path.insert(0, os.getcwd())

from config import TARGET_REPOS, RAW_DIR
from github_api import GithubDataFetcher
import time
import json

# 初始化获取器
fetcher = GithubDataFetcher()

GitHub API Rate Limit: 4992 requests remaining
Reset at: 2026-07-06 11:41:09+00:00


In [3]:
# 循环获取所有项目的 PR 数据
for idx, (owner, repo_name, desc) in enumerate(TARGET_REPOS):
    print(f"\n{'='*60}")
    print(f"项目 {idx+1}/{len(TARGET_REPOS)}: {owner}/{repo_name}")
    print(f"描述: {desc}")
    print('='*60)
    
    # 检查是否已有缓存
    if fetcher.has_data(owner, repo_name):
        print(f"数据已存在，跳过（如需重新获取，请删除 {RAW_DIR}/{owner}_{repo_name}_pulls.json）")
        continue
    
    # 获取仓库
    repo = fetcher.get_repo(owner, repo_name)
    
    # 获取 PR
    pulls = fetcher.fetch_pulls(repo, count=300)
    
    if not pulls:
        print(f"未获取到数据，跳过")
        continue
    
    # 保存原始数据
    fetcher.save_raw_data(owner, repo_name, pulls)
    
    # 防止限流，项目间间隔
    time.sleep(2)

print("\n✅ 所有项目 PR 数据获取完成！")


项目 1/5: kubernetes/kubernetes
描述: 容器编排平台，PR 审查流程严格规范
数据已存在，跳过（如需重新获取，请删除 f:\学习资料\智能软件工程实践\lab1\data\raw/kubernetes_kubernetes_pulls.json）

项目 2/5: pytorch/pytorch
描述: 深度学习框架，有 AI 生成代码和 AI Review
数据已存在，跳过（如需重新获取，请删除 f:\学习资料\智能软件工程实践\lab1\data\raw/pytorch_pytorch_pulls.json）

项目 3/5: tensorflow/tensorflow
描述: 深度学习框架，大型社区活跃
数据已存在，跳过（如需重新获取，请删除 f:\学习资料\智能软件工程实践\lab1\data\raw/tensorflow_tensorflow_pulls.json）

项目 4/5: microsoft/vscode
描述: 大型 IDE 项目，代码审查规范
数据已存在，跳过（如需重新获取，请删除 f:\学习资料\智能软件工程实践\lab1\data\raw/microsoft_vscode_pulls.json）

项目 5/5: langchain-ai/langchain
描述: LLM 应用框架，与 AI 紧密相关
数据已存在，跳过（如需重新获取，请删除 f:\学习资料\智能软件工程实践\lab1\data\raw/langchain-ai_langchain_pulls.json）

✅ 所有项目 PR 数据获取完成！


## 步骤二：获取 Code Review 和 Comments

对每个 PR 获取 Reviews、Issue Comments、Review Comments。

In [4]:
# 遍历所有 PR，获取 Reviews 和 Comments
from tqdm import tqdm
import json

for owner, repo_name, _ in TARGET_REPOS:
    print(f"\n处理 {owner}/{repo_name} 的 Reviews...")
    
    # 加载已有的 PR 数据
    pulls = fetcher.load_raw_data(owner, repo_name)
    if not pulls:
        print(f"  未找到 {owner}/{repo_name} 的数据，跳过")
        continue
    
    repo = fetcher.get_repo(owner, repo_name)
    if repo is None:
        print(f"  获取仓库 {owner}/{repo_name} 失败，跳过")
        continue
    
    # 检查哪些 PR 还没有 Review 数据
    need_fetch = [pr for pr in pulls if 'reviews' not in pr]
    if not need_fetch:
        print(f"  所有 PR 的 Review 数据已存在，跳过")
        continue
    
    print(f"  需要获取 {len(need_fetch)} 个 PR 的 Review 数据")
    
    for pr in tqdm(need_fetch):
        pr_id = pr['pr_id']
        
        # 获取 Reviews、Issue Comments 和行内 Review Comments
        reviews, issue_comments, review_comments = fetcher.fetch_reviews_and_comments(repo, pr_id)
        pr['reviews'] = reviews
        pr['issue_comments'] = issue_comments
        pr['review_comments'] = review_comments
        
        # 获取修改文件和 Commits（已包含 modified_functions）
        files, commits = fetcher.fetch_files_and_commits(repo, pr_id)
        pr['files'] = files
        pr['commits'] = commits
    
    # 保存更新后的数据
    fetcher.save_raw_data(owner, repo_name, pulls)

print("\n✅ 所有 Review 和文件数据获取完成！")


处理 kubernetes/kubernetes 的 Reviews...
  所有 PR 的 Review 数据已存在，跳过

处理 pytorch/pytorch 的 Reviews...
  所有 PR 的 Review 数据已存在，跳过

处理 tensorflow/tensorflow 的 Reviews...
  所有 PR 的 Review 数据已存在，跳过

处理 microsoft/vscode 的 Reviews...
  所有 PR 的 Review 数据已存在，跳过

处理 langchain-ai/langchain 的 Reviews...
  所有 PR 的 Review 数据已存在，跳过

✅ 所有 Review 和文件数据获取完成！


## 步骤三：数据概览

查看获取到的数据样例。

In [5]:
# 展示 PR 数据样例
from github_api import get_all_prs_data

all_pulls = get_all_prs_data()
print(f"\n共获取 {len(all_pulls)} 条 PR 数据")

# 显示前 3 条 PR 的概要信息
print("\n" + "="*60)
print("PR 数据样例（前 3 条）")
print("="*60)
for i, pr in enumerate(all_pulls[:3]):
    print(f"\n--- PR 示例 {i+1} ---")
    print(f"  PR ID: {pr['pr_id']}")
    print(f"  仓库: {pr['repo_owner']}/{pr['repo_name']}")
    print(f"  标题: {pr['title'][:80] if pr['title'] else 'N/A'}")
    print(f"  作者: {pr['author']}")
    print(f"  创建时间: {pr['created_at']}")
    print(f"  是否合并: {pr['merged']}")
    print(f"  Labels: {pr['labels']}")
    print(f"  Reviews 数量: {len(pr.get('reviews', []))}")
    print(f"  Files 数量: {len(pr.get('files', []))}")
    print(f"  Commits 数量: {len(pr.get('commits', []))}")

已加载 kubernetes/kubernetes: 300 个 PR
已加载 pytorch/pytorch: 300 个 PR
已加载 tensorflow/tensorflow: 300 个 PR
已加载 microsoft/vscode: 300 个 PR
已加载 langchain-ai/langchain: 300 个 PR

共获取 1500 条 PR 数据

PR 数据样例（前 3 条）

--- PR 示例 1 ---
  PR ID: 140072
  仓库: kubernetes/kubernetes
  标题: cleanup: fix double "the" and other comment typos in pkg
  作者: shashank-tomar0
  创建时间: 2026-06-29T07:14:13+00:00
  是否合并: False
  Labels: ['area/kubelet', 'kind/cleanup', 'sig/scheduling', 'sig/node', 'size/XS', 'kind/api-change', 'sig/auth', 'sig/apps', 'cncf-cla: yes', 'needs-ok-to-test', 'do-not-merge/release-note-label-needed', 'needs-priority', 'needs-triage', 'wg/device-management']
  Reviews 数量: 0
  Files 数量: 4
  Commits 数量: 1

--- PR 示例 2 ---
  PR ID: 140071
  仓库: kubernetes/kubernetes
  标题: [WIP] Return errors from ToleratesTaint for invalid numeric toleration values
  作者: pacoxu
  创建时间: 2026-06-29T06:37:34+00:00
  是否合并: False
  Labels: ['kind/bug', 'area/test', 'area/kubelet', 'sig/scheduling', 'sig/node', 'siz

In [6]:
# 展示 Review Comments 样例
print("="*60)
print("Review Comments 样例")
print("="*60)

sample_count = 0
for pr in all_pulls:
    reviews = pr.get('reviews', [])
    for review in reviews:
        if review.get('body') and len(review['body'].strip()) > 0:
            print(f"\n--- Review 示例 {sample_count+1} ---")
            print(f"  仓库: {pr['repo_owner']}/{pr['repo_name']} (PR #{pr['pr_id']})")
            print(f"  Reviewer: {review['reviewer']}")
            print(f"  状态: {review['state']}")
            print(f"  时间: {review['submitted_at']}")
            print(f"  内容预览: {review['body'][:200]}..." if len(review['body']) > 200 else f"  内容: {review['body']}")
            sample_count += 1
            if sample_count >= 3:
                break
    if sample_count >= 3:
        break

Review Comments 样例

--- Review 示例 1 ---
  仓库: kubernetes/kubernetes (PR #140066)
  Reviewer: HirazawaUi
  状态: COMMENTED
  时间: 2026-06-29T01:07:14+00:00
  内容: /lgtm

--- Review 示例 2 ---
  仓库: kubernetes/kubernetes (PR #140038)
  Reviewer: SergeyKanzhelev
  状态: APPROVED
  时间: 2026-06-26T17:09:23+00:00
  内容: /lgtm
/approve

--- Review 示例 3 ---
  仓库: kubernetes/kubernetes (PR #140024)
  Reviewer: BenTheElder
  状态: COMMENTED
  时间: 2026-06-25T22:11:17+00:00
  内容: /lgtm
/approve


---
## 步骤四：特征提取

从原始数据中提取所有特征，构建结构化数据集。

**新增**：提取每个 PR 修改文件的编程语言信息，包括：
- `primary_language`：该 PR 修改最多的编程语言
- `total_languages`：该 PR 涉及的语言种类数
- `languages`：该 PR 涉及的所有语言列表

In [7]:
# 构建数据集
from feature_extractor import build_dataset, FeatureExtractor

df = build_dataset()

print("\n数据集前 5 行预览:")
display(df.head())

print("\n数据集基本信息:")
print(df.info())

开始构建数据集...
已加载 kubernetes/kubernetes: 300 个 PR
已加载 pytorch/pytorch: 300 个 PR
已加载 tensorflow/tensorflow: 300 个 PR
已加载 microsoft/vscode: 300 个 PR
已加载 langchain-ai/langchain: 300 个 PR
共加载 1500 个 PR
数据集大小: 1500 行, 36 列
数据集已保存到: f:\学习资料\智能软件工程实践\lab1\data\processed\dataset.csv
统计摘要已保存到: f:\学习资料\智能软件工程实践\lab1\data\processed\summary.csv

数据集前 5 行预览:


,pr_id,repo_owner,repo_name,title,author,created_at,pr_length,title_length,files_changed,additions,...,final_review_decision,has_review,primary_language,total_languages,languages,has_ai_reviewer,ai_reviewers,ai_reviewer_count,has_ai_generated_code,ai_code_indicators
0,140072,kubernetes,kubernetes,"cleanup: fix double ""the"" and other comment ty...",shashank-tomar0,2026-06-29T07:14:13+00:00,667,56,4,4,...,None,0,Go,1,Go,1,"kubernetes-prow[bot],linux-foundation-easycla[...",3,0,
1,140071,kubernetes,kubernetes,[WIP] Return errors from ToleratesTaint for in...,pacoxu,2026-06-29T06:37:34+00:00,405,77,16,166,...,None,0,Go,2,"Go,Other(mod)",1,kubernetes-prow[bot],1,0,
2,140068,kubernetes,kubernetes,cleanup: remove stale heapster references from...,amigo-nishant,2026-06-29T05:40:44+00:00,1236,55,5,4,...,None,0,Go,1,Go,1,kubernetes-prow[bot],1,0,
3,140067,kubernetes,kubernetes,flake stabilize HPA reconciliation metric asse...,googs1025,2026-06-29T02:41:15+00:00,4966,51,1,13,...,None,0,Go,1,Go,1,kubernetes-prow[bot],1,0,
4,140066,kubernetes,kubernetes,Automated cherry pick of #139850: kubelet star...,compumike,2026-06-29T00:03:53+00:00,456,112,2,16,...,COMMENTED,1,Go,1,Go,1,kubernetes-prow[bot],1,0,



数据集基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   pr_id                    1500 non-null   int64 
 1   repo_owner               1500 non-null   object
 2   repo_name                1500 non-null   object
 3   title                    1500 non-null   object
 4   author                   1500 non-null   object
 5   created_at               1500 non-null   object
 6   pr_length                1500 non-null   int64 
 7   title_length             1500 non-null   int64 
 8   files_changed            1500 non-null   int64 
 9   additions                1500 non-null   int64 
 10  deletions                1500 non-null   int64 
 11  is_merged                1500 non-null   int64 
 12  label_count              1500 non-null   int64 
 13  labels                   1500 non-null   object
 14  modified_function_count  1500 

In [8]:
# 查看 AI 相关数据
print("="*60)
print("AI Reviewer 检测结果")
print("="*60)
ai_reviewer_prs = df[df['has_ai_reviewer'] == 1]
print(f"含 AI Reviewer 的 PR 数量: {len(ai_reviewer_prs)}")
display(ai_reviewer_prs[['repo_name', 'pr_id', 'title', 'ai_reviewers']].head(10))

print("\n" + "="*60)
print("AI 生成代码检测结果")
print("="*60)
ai_code_prs = df[df['has_ai_generated_code'] == 1]
print(f"含 AI 生成代码的 PR 数量: {len(ai_code_prs)}")
display(ai_code_prs[['repo_name', 'pr_id', 'title', 'ai_code_indicators']].head(10))

AI Reviewer 检测结果
含 AI Reviewer 的 PR 数量: 1112


,repo_name,pr_id,title,ai_reviewers
0,kubernetes,140072,"cleanup: fix double ""the"" and other comment ty...","kubernetes-prow[bot],linux-foundation-easycla[..."
1,kubernetes,140071,[WIP] Return errors from ToleratesTaint for in...,kubernetes-prow[bot]
2,kubernetes,140068,cleanup: remove stale heapster references from...,kubernetes-prow[bot]
3,kubernetes,140067,flake stabilize HPA reconciliation metric asse...,kubernetes-prow[bot]
4,kubernetes,140066,Automated cherry pick of #139850: kubelet star...,kubernetes-prow[bot]
5,kubernetes,140064,DRA: standardize mdevUUID attribute for virtua...,kubernetes-prow[bot]
6,kubernetes,140063,DRA resourceslice controller: fix update after...,kubernetes-prow[bot]
7,kubernetes,140062,storage: migrate CSIDriver volumeLifecycleMode...,"kubernetes-prow[bot],k8s-triage-robot"
8,kubernetes,140061,fix(scheduler): refresh pod status after patch...,kubernetes-prow[bot]
9,kubernetes,140059,test(scheduler): protect scheduler plugin test...,kubernetes-prow[bot]



AI 生成代码检测结果
含 AI 生成代码的 PR 数量: 344


,repo_name,pr_id,title,ai_code_indicators
47,kubernetes,140006,Pod Certificates: Node-restriction support for...,commit: claude; body: claude
64,kubernetes,139982,"WIP, DRA: Fix Numa hinting for Toplogy Manager",commit: cursor
75,kubernetes,139969,docs(apimachinery): fix malformed resourceVers...,body: claude
119,kubernetes,139918,CronJob controller: processNextWorkItem does n...,body: claude
203,kubernetes,139813,Fix DRA allocator widening config scoped to su...,commit: claude
300,pytorch,188397,[test] Fix test_torch_binomial_dtype_errors: u...,commit: co-authored-by: copilot
302,pytorch,188395,Refactor Dynamo side effect replay into regist...,commit: codex
309,pytorch,188380,docs: fix broken gpu_direct_storage tutorial U...,commit: claude
329,pytorch,188354,Remove stale xfailIfSM100OrLater on test_divis...,commit: claude; body: claude
332,pytorch,188348,[HOP] Fixes for zero-length scan dim,body: claude


---
## 步骤五：统计分析与可视化

对数据集进行全面分析并生成可视化图表。

In [9]:
# 运行所有分析
from analysis import DataAnalyzer

analyzer = DataAnalyzer(df)
analyzer.run_all_analyses()

Dataset Overview
Total PRs: 1500
Features: 36
Columns: ['pr_id', 'repo_owner', 'repo_name', 'title', 'author', 'created_at', 'pr_length', 'title_length', 'files_changed', 'additions', 'deletions', 'is_merged', 'label_count', 'labels', 'modified_function_count', 'modified_functions', 'reviewer_count', 'reviewers', 'review_count', 'review_comment_count', 'issue_comment_count', 'inline_comment_count', 'total_comment_count', 'approved_count', 'changes_requested_count', 'commented_count', 'final_review_decision', 'has_review', 'primary_language', 'total_languages', 'languages', 'has_ai_reviewer', 'ai_reviewers', 'ai_reviewer_count', 'has_ai_generated_code', 'ai_code_indicators']

----------------------------------------
PRs per Repository:
----------------------------------------
  kubernetes/kubernetes: 300 PRs (merged: 99)
  langchain-ai/langchain: 300 PRs (merged: 132)
  microsoft/vscode: 300 PRs (merged: 211)
  pytorch/pytorch: 300 PRs (merged: 8)
  tensorflow/tensorflow: 300 PRs (merge

f:\学习资料\智能软件工程实践\lab1\analysis.py:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


图表已保存: f:\学习资料\智能软件工程实践\lab1\figures\01_merge_vs_non_merge.png

分析: Review Comment 数量分布
  mean: 3.73
  median: 2.00
  min: 0
  max: 52
  std: 5.23



f:\学习资料\智能软件工程实践\lab1\analysis.py:176: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


图表已保存: f:\学习资料\智能软件工程实践\lab1\figures\02_comment_distribution.png

分析: Label 分布
  不同 Label 总数: 254
  Top 15 Labels:
    cncf-cla: yes: 295
    needs-priority: 263
    needs-triage: 235
    size: XS: 207
    open source: 187
    release-note-none: 173
    external: 170
    missing-issue-link: 166
    new-contributor: 166
    fix: 145
    internal: 130
    lgtm: 125
    topic: not user facing: 123
    approved: 117
    sig/node: 114



...

f:\学习资料\智能软件工程实践\lab1\analysis.py:374: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


图表已保存: f:\学习资料\智能软件工程实践\lab1\figures\06_ai_vs_human.png

各仓库关键指标对比
            PR_Count  Merge_Rate  Avg_Reviewers  Avg_Comments  Avg_PR_Length  Avg_Files_Changed  AI_Reviewer_Ratio  AI_Code_Ratio
repo_name                                                                                                                        
kubernetes       300       0.330          0.873         9.727       1982.200             12.223              1.000          0.017
langchain        300       0.440          0.220         0.890       3845.233              2.037              0.680          0.230
pytorch          300       0.027          0.763         4.307       1639.430             23.727              1.000          0.280
tensorflow       300       0.337          0.067         0.213        418.570              9.537              0.047          0.010
vscode           300       0.703          2.157         3.510       1370.713              6.117              0.980          0.610



f:\学习资料\智能软件工程实践\lab1\analysis.py:428: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


图表已保存: f:\学习资料\智能软件工程实践\lab1\figures\07_per_repo_comparison.png

分析: 特征相关性
相关性矩阵:
                       pr_length  title_length  files_changed  additions  deletions  reviewer_count  total_comment_count  label_count  is_merged  has_ai_reviewer  has_ai_generated_code
pr_length               1.000000      0.110659      -0.019759  -0.016160  -0.013654       -0.042963             0.008686     0.160089   0.222971        -0.184156              -0.044615
title_length            0.110659      1.000000      -0.051149  -0.039906  -0.023033        0.006997             0.061059     0.149605   0.031098         0.132131               0.095011
files_changed          -0.019759     -0.051149       1.000000   0.933925   0.091419       -0.016372             0.014640     0.007295  -0.027885         0.012121               0.042813
additions              -0.016160     -0.039906       0.933925   1.000000   0.018057       -0.010363             0.021016    -0.015572  -0.025264         0.020060               0.

f:\学习资料\智能软件工程实践\lab1\analysis.py:473: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 实验思考题

### 1. Pull Request 与 Commit 有何区别？

- **Pull Request**：是一个代码合并请求，包含多个 Commit、讨论、Review 等，是一个协作单元
- **Commit**：是代码仓库的一个快照，代表一次代码修改的原子操作
- 一个 PR 可以包含多个 Commit，而一个 Commit 只能属于一个 PR（或不在任何 PR 中）

### 2. 为什么 Code Review 能够提高软件质量？

- **发现缺陷**：通过多人审查发现潜在 Bug 和安全漏洞
- **统一规范**：确保代码风格和架构设计的一致性
- **知识共享**：促进团队间的技术交流和知识传递
- **提前发现**：在合并前发现问题，降低修复成本

### 3. Merge Prediction 可以应用于哪些实际场景？

- **CI/CD 自动化**：预测 PR 合并概率，优化 CI 资源分配
- **PR 优先级排序**：帮助维护者更快处理高合并概率的 PR
- **开发者反馈**：给提交者提供 PR 被合并的概率预估
- **社区管理**：分析开源项目中影响合并的关键因素

### 4. AI Reviewer 与 Human Reviewer 的审查方式可能有哪些差异？

- **AI Reviewer**：速度快、覆盖全、模板化、一致性高，但缺乏对项目上下文的深入理解
- **Human Reviewer**：理解项目架构和业务逻辑，能发现语义问题，但速度慢、受精力影响
- AI 更适合检查代码规范、常见 bug 模式；Human 更适合架构评审、设计决策

### 5. 数据集是否存在类别不平衡问题？如何解决？

- **分析**：Merge 的 PR 通常远多于 Non-Merge 的 PR，存在类别不平衡
- **解决方法**：
  1. 欠采样（Undersampling）：随机减少多数类样本
  2. 过采样（Oversampling）：复制少数类样本或使用 SMOTE
  3. 加权损失函数：在训练时给少数类更高的权重
  4. 使用 AUC-ROC 等对不平衡不敏感的评估指标

---
## 参考资料

- [GitHub REST API 文档](https://docs.github.com/en/rest/pulls/pulls)
- [PyGithub 文档](https://pygithub.readthedocs.io/)
- [pandas 文档](https://pandas.pydata.org/)
- [matplotlib 文档](https://matplotlib.org/)